<a href="https://colab.research.google.com/github/SkSania/Alfido-task1-IRIS-dataset-/blob/main/Task_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# ==========================================
# 1. LOAD & PREPROCESS DATA
# ==========================================
csv_path = "data[1].csv"
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"'{csv_path}' not found.")
df = pd.read_csv(csv_path)

# Identify target (assuming 'price' is present)
target_col = [col for col in df.columns if 'price' in col.lower()][0]
X = df.drop(columns=[target_col])
y = df[target_col]

# Define feature types
num_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_features = X.select_dtypes(include=['object', 'category']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# ==========================================
# 2. PIPELINE & TRAINING
# ==========================================
preprocessor = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_features),
    ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
])

# Using RandomForest for simplicity
model = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', RandomForestRegressor(random_state=42))])
model.fit(X_train, y_train)

# Evaluation
preds = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, preds))
print(f"Random Forest RMSE: {rmse:,.2f}")

# ==========================================
# 3. SAVE MODEL & INFERENCE
# ==========================================
joblib.dump(model, "house_model.joblib")
print("Model saved as house_model.joblib")

Random Forest RMSE: 984,735.54
Model saved as house_model.joblib
